# Exp.01 — PGR²M residual-depth temporal-detail experiment

## Reviewer question

**Does the residual RVQ actually restore temporal motion detail beyond the pose-guided representation?**

This experiment freezes the retrained tokenizer and progressively enables the
six residual quantizers:

| Depth | Reconstruction path |
|---:|---|
| 0 | pose-guided latent only |
| 1 | pose + residual stage 1 |
| 2 | pose + residual stages 1–2 |
| 3 | pose + residual stages 1–3 |
| 4 | pose + residual stages 1–4 |
| 5 | pose + residual stages 1–5 |
| 6 | pose + all six residual stages (full tokenizer) |

The checkpoint is **not retrained** in this experiment. The default checkpoint
is the frozen `net_best_fid.pth` from the completed 300k tokenizer run.

### Primary metrics

All metrics are **lower is better**.

- **MPJPE**: root-aligned joint-position reconstruction error.
- **PA-MPJPE**: frame-wise Procrustes-aligned joint-position error.
- **Velocity error**: first-difference error on root-relative joints.
- **Acceleration error**: second-difference error on root-relative joints.
- **Jerk error**: third-difference error on root-relative joints.
- **HF spectral error**: L1 error between normalized high-frequency magnitude
  spectra of root-relative joint trajectories.
- **HF energy-ratio error**: absolute error in the fraction of spectral energy
  contained in the upper half of non-DC frequency bins.
- **Root-trajectory velocity error**: first-difference error of the pelvis/root
  trajectory.

Discrete derivatives are reported **per frame** (not multiplied by FPS), so the
temporal measures remain directly comparable to the repository's existing
acceleration metric convention.

### Statistical reporting

The notebook stores per-clip metrics and computes **paired bootstrap 95% CIs
with 1000 resamples** for:

1. every residual depth;
2. full tokenizer versus pose-only;
3. each additional residual stage versus the preceding depth.

The main evaluation split is **HumanML3D test** with deterministic ordering,
seed 123, and `num_workers=0`.


In [ ]:
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime
import sys, os, json, yaml, hashlib, random, math, gc
import numpy as np
import pandas as pd
import torch

PGR = Path(".")
RESULTS = PGR / "revision_results" / "exp01_residual_depth"
RESULTS.mkdir(parents=True, exist_ok=True)

EXPECTED_PY = Path("venv/bin/python")
print("Python executable :", sys.executable)
print("Expected env      :", EXPECTED_PY)
print("Torch             :", torch.__version__)
print("CUDA available    :", torch.cuda.is_available())
print("GPU               :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for Exp.01.")

# Reproducibility
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Full reviewer experiment by default.
# Set an integer such as 20 only for a quick pilot.
MAX_BATCHES = None

# 1000 paired bootstrap resamples as frozen reviewer-analysis protocol.
N_BOOT = 1000
BOOT_SEED = 2027

# HumanML3D is 20 FPS, but temporal derivatives below remain per-frame.
FPS = 20

# Runtime compatibility required by the public PGR²M source under modern NumPy.
if not hasattr(np, "float"):
    np.float = float

if str(PGR) not in sys.path:
    sys.path.insert(0, str(PGR))
os.chdir(PGR)

print("Results directory:", RESULTS)


## 1. Locate the completed 300k run and freeze the evaluation checkpoint


In [ ]:
launch_record_path = PGR / "revision_logs" / "exp00t_latest_launch.json"
audit_summary_path = PGR / "revision_results" / "exp00u_300k_summary.json"

if not launch_record_path.exists():
    raise FileNotFoundError(launch_record_path)

launch = json.loads(launch_record_path.read_text())
run_root = Path(launch["output_root"]) / launch["exp_name"]
runs = sorted([p for p in run_root.glob("*") if p.is_dir()], key=lambda p: p.stat().st_mtime)

if not runs:
    raise RuntimeError(f"No completed production run found under {run_root}")

RUN_DIR = runs[-1]
ARGS_FILE = RUN_DIR / "arguments.yaml"
CKPT = RUN_DIR / "net_best_fid.pth"
LAST_CKPT = RUN_DIR / "net_last.pth"

if not ARGS_FILE.exists():
    raise FileNotFoundError(ARGS_FILE)
if not CKPT.exists():
    raise FileNotFoundError(CKPT)
if not LAST_CKPT.exists():
    raise FileNotFoundError(LAST_CKPT)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

best_obj = torch.load(CKPT, map_location="cpu", weights_only=False)
last_obj = torch.load(LAST_CKPT, map_location="cpu", weights_only=False)

print("Run directory       :", RUN_DIR)
print("Evaluation checkpoint:", CKPT)
print("best-FID nb_iter    :", best_obj.get("nb_iter"))
print("net_last nb_iter    :", last_obj.get("nb_iter"))
print("Checkpoint SHA256   :", sha256(CKPT))

if last_obj.get("nb_iter") != 300000:
    raise RuntimeError("The selected production run does not contain a 300k net_last checkpoint.")

# Exp00U established net_best_fid as the main frozen reviewer checkpoint.
if best_obj.get("nb_iter") is None:
    raise RuntimeError("net_best_fid.pth has no nb_iter metadata.")

del last_obj
gc.collect()


## 2. Reconstruct the tokenizer exactly from the frozen training arguments


In [ ]:
raw = ARGS_FILE.read_text()
try:
    arg_dict = json.loads(raw)
except Exception:
    arg_dict = yaml.safe_load(raw)

args = SimpleNamespace(**arg_dict)

from models.pg_tokenizer import PoseGuidedTokenizer

net = PoseGuidedTokenizer(
    args,
    args.nb_code,
    args.code_dim,
    args.output_emb_width,
    args.down_t,
    args.stride_t,
    args.width,
    args.depth,
    args.dilation_growth_rate,
    args.vq_act,
    args.vq_norm,
    num_quantizers=args.rvq_num_quantizers,
    shared_codebook=args.rvq_shared_codebook,
    quantize_dropout_prob=args.rvq_quantize_dropout_prob,
    quantize_dropout_cutoff_index=args.rvq_quantize_dropout_cutoff_index,
    rvq_nb_code=args.rvq_nb_code,
    mu=args.rvq_mu,
    residual_ratio=args.rvq_residual_ratio,
    vq_loss_beta=args.rvq_vq_loss_beta,
    quantizer_type=args.rvq_quantizer_type,
    params_soft_ent_loss=args.params_soft_ent_loss,
    use_ema=(not args.unuse_ema),
    init_method=args.rvq_init_method,
)

missing, unexpected = net.load_state_dict(best_obj["net"], strict=False)
print("Missing state keys   :", missing)
print("Unexpected state keys:", unexpected)

if missing or unexpected:
    raise RuntimeError("Checkpoint/model state_dict mismatch.")

net = net.cuda().eval()
print("RVQ stages:", net.rvq.num_quantizers)
print("RVQ codes :", net.rvq_nb_code)

if net.rvq.num_quantizers != 6:
    raise RuntimeError(f"Expected 6 RVQ stages, got {net.rvq.num_quantizers}.")

del best_obj
torch.cuda.empty_cache()


## 3. Build the deterministic HumanML3D **test** loader

The reconstructed loader is used only as the data interface. Text fields are
loaded because they are part of the public evaluator tuple, but Exp.01 does not
use text semantics.


In [ ]:
from dataset import dataset_TM_eval
from utils.word_vectorizer import WordVectorizer

w_vectorizer = WordVectorizer("./glove", "our_vab")

loader_kwargs = dict(
    dataset_name="t2m",
    is_test=True,
    batch_size=32,
    w_vectorizer=w_vectorizer,
    codebook_size=args.nb_code,
    val_shuffle=False,
    unit_length=2 ** args.down_t,
    num_workers=0,
    use_keywords=getattr(args, "use_keywords", True),
    use_word_only=False,
    meta_dir=getattr(args, "meta_dir", None),
    codes_folder_name=getattr(args, "codes_folder_name", "codes"),
)

print("Loader kwargs:", loader_kwargs)

test_loader = dataset_TM_eval.DATALoader(**loader_kwargs)
print("Test dataset size:", len(test_loader.dataset))
print("Test batches      :", len(test_loader))

# One tuple-contract check.
batch0 = next(iter(test_loader))
print("Tuple fields      :", len(batch0))
print("Motion shape      :", tuple(batch0[4].shape))
print("Length shape      :", tuple(batch0[5].shape))
print("Code shape        :", tuple(batch0[8].shape))

if len(batch0) < 9:
    raise RuntimeError("Unexpected reconstructed test-loader tuple.")


## 4. Metric definitions

For temporal-detail metrics, the pelvis/root is removed **frame by frame**
before taking local-joint derivatives. A separate root-trajectory velocity
metric preserves trajectory information.

HF spectral error uses root-relative trajectories, subtracts each
joint-coordinate temporal mean, applies RFFT, removes DC, normalizes the
non-DC magnitude spectrum per joint-coordinate, and evaluates the upper half
of the remaining frequency bins.


In [ ]:
from utils.motion_process import recover_from_ric
from utils.metrics import calc_pampjpe

EPS = 1e-8

def root_relative(xyz):
    # xyz: T x J x 3
    return xyz - xyz[:, 0:1, :]

def mean_joint_norm(x):
    # x: T x J x 3
    return torch.linalg.norm(x, dim=-1).mean()

def discrete_error(pred, gt, order):
    if order == 1:
        dp = pred[1:] - pred[:-1]
        dg = gt[1:] - gt[:-1]
    elif order == 2:
        dp = pred[:-2] - 2 * pred[1:-1] + pred[2:]
        dg = gt[:-2] - 2 * gt[1:-1] + gt[2:]
    elif order == 3:
        dp = pred[3:] - 3 * pred[2:-1] + 3 * pred[1:-2] - pred[:-3]
        dg = gt[3:] - 3 * gt[2:-1] + 3 * gt[1:-2] - gt[:-3]
    else:
        raise ValueError(order)
    return mean_joint_norm(dp - dg)

def root_velocity_error(pred_xyz, gt_xyz):
    dp = pred_xyz[1:, 0, :] - pred_xyz[:-1, 0, :]
    dg = gt_xyz[1:, 0, :] - gt_xyz[:-1, 0, :]
    return torch.linalg.norm(dp - dg, dim=-1).mean()

def hf_spectral_metrics(pred_rr, gt_rr):
    # T x J x 3
    pred = pred_rr - pred_rr.mean(dim=0, keepdim=True)
    gt = gt_rr - gt_rr.mean(dim=0, keepdim=True)

    pmag = torch.abs(torch.fft.rfft(pred, dim=0))[1:]  # remove DC
    gmag = torch.abs(torch.fft.rfft(gt, dim=0))[1:]

    if pmag.shape[0] < 2:
        return torch.tensor(float("nan"), device=pred.device), torch.tensor(float("nan"), device=pred.device)

    pnorm = pmag / (pmag.sum(dim=0, keepdim=True) + EPS)
    gnorm = gmag / (gmag.sum(dim=0, keepdim=True) + EPS)

    hf_start = int(math.ceil(pmag.shape[0] / 2))
    spec_l1 = torch.abs(pnorm[hf_start:] - gnorm[hf_start:]).mean()

    penergy = pmag.square()
    genergy = gmag.square()
    pratio = penergy[hf_start:].sum(dim=0) / (penergy.sum(dim=0) + EPS)
    gratio = genergy[hf_start:].sum(dim=0) / (genergy.sum(dim=0) + EPS)
    energy_ratio_err = torch.abs(pratio - gratio).mean()

    return spec_l1, energy_ratio_err

@torch.no_grad()
def clip_metrics(pred_motion, gt_motion, length, inv_transform):
    L = int(length)
    if L < 8:
        return None

    pred_np = inv_transform(pred_motion[:L].detach().cpu().numpy())
    gt_np = inv_transform(gt_motion[:L].detach().cpu().numpy())

    pred_dn = torch.from_numpy(np.asarray(pred_np)).float().cuda()
    gt_dn = torch.from_numpy(np.asarray(gt_np)).float().cuda()

    if not torch.isfinite(pred_dn).all() or not torch.isfinite(gt_dn).all():
        return None

    pred_xyz = recover_from_ric(pred_dn, 22)
    gt_xyz = recover_from_ric(gt_dn, 22)

    pred_rr = root_relative(pred_xyz)
    gt_rr = root_relative(gt_xyz)

    mpjpe = mean_joint_norm(pred_rr - gt_rr)
    pampjpe = calc_pampjpe(pred_xyz, gt_xyz).mean()

    vel = discrete_error(pred_rr, gt_rr, 1)
    acc = discrete_error(pred_rr, gt_rr, 2)
    jerk = discrete_error(pred_rr, gt_rr, 3)
    root_vel = root_velocity_error(pred_xyz, gt_xyz)
    hf_spec, hf_energy = hf_spectral_metrics(pred_rr, gt_rr)

    return {
        "mpjpe": float(mpjpe.item()),
        "pampjpe": float(pampjpe.item()),
        "vel_err": float(vel.item()),
        "accel_err": float(acc.item()),
        "jerk_err": float(jerk.item()),
        "root_vel_err": float(root_vel.item()),
        "hf_spec_err": float(hf_spec.item()),
        "hf_energy_ratio_err": float(hf_energy.item()),
    }

print("[OK] metric definitions")


## 5. Sanity-check the residual-depth control

`force_dropout_index=k-1` keeps quantizers `0...k-1` and removes later stages.
Depth 6 uses the normal full path with no forced dropout.


In [ ]:
def depth_forward(net, code_ds, motion, depth):
    if depth == 0:
        pred, _, _ = net(
            code_ds,
            motion,
            detach_p_latent=True,
            drop_out_residual_quantization=True,
        )
        return pred

    if depth < 6:
        pred, _, _ = net(
            code_ds,
            motion,
            detach_p_latent=True,
            drop_out_residual_quantization=False,
            force_dropout_index=depth - 1,
        )
        return pred

    if depth == 6:
        pred, _, _ = net(
            code_ds,
            motion,
            detach_p_latent=True,
            drop_out_residual_quantization=False,
            force_dropout_index=-1,
        )
        return pred

    raise ValueError(depth)

# A one-batch output-shape and finite-value check.
motion0 = batch0[4].cuda().float()
codes0 = batch0[8][:, :: 2 ** args.down_t, :].cuda().float()

with torch.no_grad():
    for depth in range(7):
        pred0 = depth_forward(net, codes0, motion0, depth)
        print(depth, tuple(pred0.shape), bool(torch.isfinite(pred0).all()))
        if pred0.shape != motion0.shape:
            raise RuntimeError(f"Depth {depth}: output shape mismatch.")
        if not torch.isfinite(pred0).all():
            raise RuntimeError(f"Depth {depth}: non-finite output.")

del motion0, codes0, pred0, batch0
torch.cuda.empty_cache()
print("[OK] residual-depth control")


## 6. Full test-set evaluation

This is the main Exp.01 run. With `MAX_BATCHES=None`, every test batch is used.
Per-clip rows are saved incrementally after each batch so results survive a
notebook interruption.


In [ ]:
PER_CLIP_CSV = RESULTS / "exp01_per_clip.csv"
PROGRESS_JSON = RESULTS / "exp01_progress.json"

depth_labels = {
    0: "Pose-only",
    1: "+R1",
    2: "+R1:R2",
    3: "+R1:R3",
    4: "+R1:R4",
    5: "+R1:R5",
    6: "Full R1:R6",
}

rows = []
skipped_nonfinite = []
processed_clips = 0

net.eval()

for batch_idx, batch in enumerate(test_loader):
    if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
        break

    motion = batch[4].cuda().float()
    lengths = batch[5]
    names = batch[7]
    codes = batch[8][:, :: 2 ** args.down_t, :].cuda().float()

    # Reject only samples whose input itself is non-finite.
    finite_sample = torch.isfinite(motion).reshape(motion.shape[0], -1).all(dim=1)

    for depth in range(7):
        with torch.no_grad():
            pred = depth_forward(net, codes, motion, depth)

        for i in range(motion.shape[0]):
            clip_name = str(names[i])

            if not bool(finite_sample[i].item()):
                if depth == 0:
                    skipped_nonfinite.append(clip_name)
                continue

            metrics = clip_metrics(
                pred[i],
                motion[i],
                int(lengths[i]),
                test_loader.dataset.inv_transform,
            )
            if metrics is None:
                if depth == 0:
                    skipped_nonfinite.append(clip_name)
                continue

            row = {
                "clip": clip_name,
                "length": int(lengths[i]),
                "depth": depth,
                "label": depth_labels[depth],
                **metrics,
            }
            rows.append(row)

        del pred

    processed_clips += int(finite_sample.sum().item())

    # Incremental persistence.
    pd.DataFrame(rows).to_csv(PER_CLIP_CSV, index=False)
    PROGRESS_JSON.write_text(json.dumps({
        "batch_idx": batch_idx,
        "processed_finite_clips": processed_clips,
        "rows": len(rows),
        "skipped_nonfinite_unique": sorted(set(skipped_nonfinite)),
        "max_batches": MAX_BATCHES,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }, indent=2))

    if batch_idx % 10 == 0 or batch_idx == len(test_loader) - 1:
        print(
            f"[RUN] batch {batch_idx + 1}/{len(test_loader)} | "
            f"finite clips={processed_clips} | rows={len(rows)} | "
            f"skipped={len(set(skipped_nonfinite))}"
        )

    del motion, codes
    torch.cuda.empty_cache()

per_clip = pd.DataFrame(rows)

print("\n=== EXP.01 EVALUATION COMPLETE ===")
print("Rows                    :", len(per_clip))
print("Unique clips            :", per_clip["clip"].nunique())
print("Rows per valid clip     :", len(per_clip) / max(per_clip["clip"].nunique(), 1))
print("Skipped nonfinite clips :", sorted(set(skipped_nonfinite)))
print("Saved                   :", PER_CLIP_CSV)

if per_clip.empty:
    raise RuntimeError("No valid Exp.01 rows were produced.")

# Every retained clip must have all seven depth conditions.
counts = per_clip.groupby("clip")["depth"].nunique()
bad = counts[counts != 7]
print("Incomplete clip count   :", len(bad))
if len(bad):
    raise RuntimeError("Some clips do not have all seven residual-depth conditions.")


## 7. Mean metrics + paired bootstrap 95% confidence intervals


In [ ]:
metric_cols = [
    "mpjpe",
    "pampjpe",
    "vel_err",
    "accel_err",
    "jerk_err",
    "root_vel_err",
    "hf_spec_err",
    "hf_energy_ratio_err",
]

rng = np.random.default_rng(BOOT_SEED)

def bootstrap_mean_ci(values, n_boot=N_BOOT, rng=None):
    values = np.asarray(values, dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan

    local_rng = rng if rng is not None else np.random.default_rng(BOOT_SEED)
    boots = np.empty(n_boot, dtype=np.float64)
    n = len(values)

    for b in range(n_boot):
        idx = local_rng.integers(0, n, size=n)
        boots[b] = values[idx].mean()

    return (
        float(values.mean()),
        float(np.quantile(boots, 0.025)),
        float(np.quantile(boots, 0.975)),
    )

summary_rows = []

for depth in range(7):
    sub = per_clip[per_clip["depth"] == depth]
    for metric in metric_cols:
        mean, lo, hi = bootstrap_mean_ci(sub[metric].to_numpy(), rng=rng)
        summary_rows.append({
            "depth": depth,
            "label": depth_labels[depth],
            "metric": metric,
            "mean": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_clips": len(sub),
        })

summary = pd.DataFrame(summary_rows)
SUMMARY_CSV = RESULTS / "exp01_summary_bootstrap.csv"
summary.to_csv(SUMMARY_CSV, index=False)

pivot = summary.pivot(index=["depth", "label"], columns="metric", values="mean").reset_index()
print(pivot.to_string(index=False))
print("\nSaved:", SUMMARY_CSV)


## 8. Paired improvement: full tokenizer versus pose-only

For every lower-is-better metric,

\[
\Delta = \text{PoseOnly error} - \text{Full error}.
\]

Therefore **positive Δ means the residual stack improves reconstruction**.


In [ ]:
paired = per_clip.pivot(index="clip", columns="depth", values=metric_cols)
paired_rows = []

rng_pair = np.random.default_rng(BOOT_SEED + 1)

for metric in metric_cols:
    pose_vals = paired[(metric, 0)].to_numpy(dtype=np.float64)
    full_vals = paired[(metric, 6)].to_numpy(dtype=np.float64)
    improvement = pose_vals - full_vals

    mean, lo, hi = bootstrap_mean_ci(improvement, rng=rng_pair)
    paired_rows.append({
        "comparison": "PoseOnly_minus_Full",
        "metric": metric,
        "mean_improvement": mean,
        "ci95_low": lo,
        "ci95_high": hi,
        "n_clips": len(improvement),
        "fraction_improved": float(np.mean(improvement > 0)),
    })

full_vs_pose = pd.DataFrame(paired_rows)
FULL_VS_POSE_CSV = RESULTS / "exp01_full_vs_pose_paired_bootstrap.csv"
full_vs_pose.to_csv(FULL_VS_POSE_CSV, index=False)

print(full_vs_pose.to_string(index=False))
print("\nSaved:", FULL_VS_POSE_CSV)


## 9. Incremental gain of each residual stage

For each `k=1...6`:

\[
\Delta_k = E_{k-1} - E_k,
\]

so a positive value means adding stage `k` reduces error.


In [ ]:
stage_rows = []
rng_stage = np.random.default_rng(BOOT_SEED + 2)

for depth in range(1, 7):
    for metric in metric_cols:
        prev_vals = paired[(metric, depth - 1)].to_numpy(dtype=np.float64)
        cur_vals = paired[(metric, depth)].to_numpy(dtype=np.float64)
        gain = prev_vals - cur_vals

        mean, lo, hi = bootstrap_mean_ci(gain, rng=rng_stage)
        stage_rows.append({
            "stage_added": depth,
            "comparison": f"depth{depth-1}_minus_depth{depth}",
            "metric": metric,
            "mean_gain": mean,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_clips": len(gain),
            "fraction_improved": float(np.mean(gain > 0)),
        })

stage_gain = pd.DataFrame(stage_rows)
STAGE_GAIN_CSV = RESULTS / "exp01_incremental_stage_gain.csv"
stage_gain.to_csv(STAGE_GAIN_CSV, index=False)

main_stage = stage_gain[
    stage_gain["metric"].isin(["mpjpe", "vel_err", "accel_err", "jerk_err", "hf_spec_err"])
]
print(main_stage.to_string(index=False))
print("\nSaved:", STAGE_GAIN_CSV)


## 10. Reviewer-facing table

Position and discrete-derivative metrics are shown in **millimeters per frame
order** for readability:

- MPJPE / PA-MPJPE: mm
- velocity error: mm/frame
- acceleration error: mm/frame²
- jerk error: mm/frame³

HF spectral error remains unitless.


In [ ]:
table = pivot[[
    "depth", "label",
    "mpjpe", "pampjpe",
    "vel_err", "accel_err", "jerk_err",
    "hf_spec_err",
]].copy()

for c in ["mpjpe", "pampjpe", "vel_err", "accel_err", "jerk_err"]:
    table[c] = table[c] * 1000.0

table = table.rename(columns={
    "mpjpe": "MPJPE_mm",
    "pampjpe": "PA_MPJPE_mm",
    "vel_err": "VelErr_mm_per_frame",
    "accel_err": "AccelErr_mm_per_frame2",
    "jerk_err": "JerkErr_mm_per_frame3",
    "hf_spec_err": "HF_Spectral_Error",
})

TABLE_CSV = RESULTS / "exp01_reviewer_table.csv"
table.to_csv(TABLE_CSV, index=False)

print(table.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\nSaved:", TABLE_CSV)


## 11. Figures

Separate figures are saved for the primary depth trends.


In [ ]:
import matplotlib.pyplot as plt

plot_specs = [
    ("mpjpe", 1000.0, "MPJPE (mm)", "exp01_depth_mpjpe.png"),
    ("vel_err", 1000.0, "Velocity error (mm/frame)", "exp01_depth_velocity.png"),
    ("accel_err", 1000.0, "Acceleration error (mm/frame²)", "exp01_depth_acceleration.png"),
    ("jerk_err", 1000.0, "Jerk error (mm/frame³)", "exp01_depth_jerk.png"),
    ("hf_spec_err", 1.0, "HF spectral error", "exp01_depth_hf_spectral.png"),
]

for metric, scale, ylabel, fname in plot_specs:
    s = summary[summary["metric"] == metric].sort_values("depth")
    x = s["depth"].to_numpy()
    y = s["mean"].to_numpy() * scale
    lo = s["ci95_low"].to_numpy() * scale
    hi = s["ci95_high"].to_numpy() * scale

    fig = plt.figure(figsize=(7.2, 4.6))
    ax = fig.add_subplot(111)
    ax.plot(x, y, marker="o")
    ax.fill_between(x, lo, hi, alpha=0.2)
    ax.set_xlabel("Residual depth")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(7))
    ax.set_xticklabels([depth_labels[d] for d in range(7)], rotation=25, ha="right")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()

    path = RESULTS / fname
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved:", path)


## 12. Provenance and final status


In [ ]:
PROVENANCE = RESULTS / "exp01_provenance.json"
FINAL_JSON = RESULTS / "exp01_final_summary.json"

prov = {
    "experiment": "Exp.01 residual-depth temporal-detail",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "bootstrap_seed": BOOT_SEED,
    "bootstrap_resamples": N_BOOT,
    "max_batches": MAX_BATCHES,
    "split": "HumanML3D test",
    "fps_reference": FPS,
    "temporal_derivative_units": "per-frame discrete differences; no FPS scaling",
    "run_dir": str(RUN_DIR),
    "checkpoint": str(CKPT),
    "checkpoint_nb_iter": int(torch.load(CKPT, map_location="cpu", weights_only=False).get("nb_iter")),
    "checkpoint_sha256": sha256(CKPT),
    "num_rvq_stages": int(net.rvq.num_quantizers),
    "depth_definition": {
        "0": "pose only",
        "1": "residual stage 1",
        "2": "residual stages 1-2",
        "3": "residual stages 1-3",
        "4": "residual stages 1-4",
        "5": "residual stages 1-5",
        "6": "all six residual stages",
    },
    "root_alignment": "frame-wise root-relative joints for MPJPE and temporal-detail metrics",
    "hf_spectral_definition": (
        "root-relative trajectories; temporal mean removed; RFFT; DC removed; "
        "per joint-coordinate non-DC magnitude normalized; upper half of remaining "
        "frequency bins; mean absolute spectral difference"
    ),
    "valid_clip_count": int(per_clip["clip"].nunique()),
    "skipped_nonfinite_clips": sorted(set(skipped_nonfinite)),
}
PROVENANCE.write_text(json.dumps(prov, indent=2))

final = {
    "provenance": prov,
    "full_vs_pose": full_vs_pose.to_dict(orient="records"),
    "reviewer_table": table.to_dict(orient="records"),
}
FINAL_JSON.write_text(json.dumps(final, indent=2))

required_outputs = [
    PER_CLIP_CSV,
    SUMMARY_CSV,
    FULL_VS_POSE_CSV,
    STAGE_GAIN_CSV,
    TABLE_CSV,
    PROVENANCE,
    FINAL_JSON,
]

FINALIZED = all(p.exists() and p.stat().st_size > 0 for p in required_outputs)

print("=== EXP.01 FINAL STATUS ===")
print("Checkpoint                 :", CKPT)
print("Checkpoint iteration       :", prov["checkpoint_nb_iter"])
print("Test clips evaluated       :", prov["valid_clip_count"])
print("Skipped nonfinite clips    :", len(prov["skipped_nonfinite_clips"]))
print("Residual depths completed  :", sorted(per_clip["depth"].unique().tolist()))
print("Bootstrap resamples        :", N_BOOT)
print("Required outputs complete  :", FINALIZED)
print("Results directory          :", RESULTS)
print("FINALIZED                  :", FINALIZED)

if FINALIZED:
    print("\nDECISION: RESIDUAL-DEPTH TEMPORAL-DETAIL EXPERIMENT COMPLETE.")
    print("Send the final status + reviewer-facing table + full-vs-pose paired table.")
else:
    raise RuntimeError("Exp.01 outputs are incomplete.")
